# Drive Structure (Colab)

To run this notebook in Google Colab, ensure your Google Drive has the following layout and exact folder names:

- Base repo folder: `/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject`
- Checkpoint file: `checkpoints/eomt_cityscapes.bin`
- Datasets under: `dataset/anomaly/Validation_Dataset/`
- Required dataset folders (exact names):
  - `RoadAnomaly`
  - `RoadAnomaly21`
  - `RoadObsticle21`
  - `fs_static`
  - `FS_LostFound_full`
- Each dataset must contain:
  - `images/` (RGB input images)
  - `labels_masks/` (binary ground-truth masks; 255 for anomaly where applicable)

Notes:
- Avoid trailing spaces in any folder name.
- Mount path in Colab is `/content/drive/MyDrive`.
- This notebook focuses on Colab-only setup; WSL/Conda instructions were removed.

## Colab setup
- Runtime → Change runtime type → GPU (T4 or better).
- Mount Drive and install the deps with the next cell.
- Verify GPU/CUDA in the following cell.
- All paths assume Drive is mounted at `/content/drive`.

## Checkpoint and datasets
- Checkpoint at `checkpoints/eomt_cityscapes.bin`.
- Datasets under `dataset/anomaly/Validation_Dataset/` with subfolders: RoadAnomaly, RoadAnomaly21, RoadObsticle21, fs_static, FS_LostFound_full.
- Each dataset must have `images/` and `labels_masks/`.


In [ ]:
# Mount Drive and install deps
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Drive mounted')
except Exception as e:
    print('Not in Colab or mount not needed:', e)

!python3 -m pip install --quiet lightning scikit-learn

In [ ]:
import os
from pathlib import Path

base = Path('/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ')

# First check if base repo exists
print(f"Base repo path: {base}")
print(f"Base exists: {base.exists()}")

if base.exists():
    # List what's actually in the repo
    print(f"\nContents of repo root:")
    try:
        items = [item.name for item in base.iterdir()]
        for item in sorted(items)[:10]:
            print(f"  - {item}")
    except Exception as e:
        print(f"Error: {e}")

# Now check specific paths
datasets = [
    'RoadAnomaly',
    'RoadAnomaly21',
    'RoadObsticle21',
    'fs_static',
    'FS_LostFound_full',
]

paths_to_check = [
    base / 'checkpoints' / 'eomt_cityscapes.bin',
    base / 'dataset' / 'anomaly' / 'Validation_Dataset',
]
for d in datasets:
    root = base / 'dataset' / 'anomaly' / 'Validation_Dataset' / d
    paths_to_check.extend([
        root,
        root / 'images',
        root / 'labels_masks',
    ])

print('\nDrive structure check:')
for p in paths_to_check:
    exists_pathlib = p.exists()
    exists_os = os.path.exists(str(p))
    status = 'OK' if exists_pathlib else 'MISSING'
    print(f'{status:8} {p}')
    if not exists_pathlib and exists_os:
        print(f"         ^ (os.path sees it but Path doesn't)")

## Find repo location (Diagnostics)
If the structure check shows MISSING files, run this cell after mounting Drive to locate where your repo actually is.

In [ ]:
import os
from pathlib import Path

# Check if Drive is mounted
drive_root = Path('/content/drive')
print(f"Drive root exists? {drive_root.exists()}")

if drive_root.exists():
    mydrive = drive_root / 'MyDrive'
    print(f"MyDrive exists? {mydrive.exists()}")
    
    if mydrive.exists():
        print("\nListing top-level folders in MyDrive:")
        try:
            items = sorted([item.name for item in mydrive.iterdir() if item.is_dir()])
            for item in items[:20]:
                # Show with quotes to see spaces
                print(f"  - '{item}'")
        except Exception as e:
            print(f"Error listing: {e}")
        
        # Search for the repo folder
        print("\nSearching for 'MaskArchitecture' folders...")
        candidates = list(mydrive.glob("*Mask*"))
        if candidates:
            for c in candidates:
                print(f"Found: '{c}'")
                print(f"  Exact name: '{c.name}'")
                print(f"  Name length: {len(c.name)}")
                # Check if it contains eomt folder
                if (c / 'eomt').exists():
                    print(f"  ✓ Contains eomt/ folder")
                if (c / 'checkpoints').exists():
                    print(f"  ✓ Contains checkpoints/ folder")
        else:
            print("No folders matching '*Mask*' found")
else:
    print("Drive is not mounted! Run the Drive mount cell first.")

In [ ]:
# Verify GPU/CUDA
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## Single test run (quick check)
Runs one evaluation on RoadObsticle21 with MaxEntropy at temp=0.5; outputs to console and results.txt.


In [ ]:
import os
from pathlib import Path

# Debug: verify paths before running test
repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")
print("Repo base:", repo_base)
print("Repo exists?", repo_base.exists())
print("Script exists?", (repo_base / "eomt" / "evalAnomaly.py").exists())
print("Checkpoint exists?", (repo_base / "checkpoints" / "eomt_cityscapes.bin").exists())
print("Dataset exists?", (repo_base / "dataset" / "anomaly" / "Validation_Dataset" / "RoadObsticle21").exists())
print("Dataset images?", (repo_base / "dataset" / "anomaly" / "Validation_Dataset" / "RoadObsticle21" / "images").exists())


In [ ]:
import subprocess, os, shutil
from pathlib import Path

repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")

CKPT = str(repo_base / "checkpoints" / "eomt_cityscapes.bin")
INPUT = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset" / "RoadObsticle21")
METHOD = "maxentropy"
TEMP = "0.5"

cmd = [
    "python3", str(repo_base / "eomt" / "evalAnomaly.py"),
    "--ckpt", CKPT,
    "--input", INPUT,
    "--method", METHOD,
    "--temp", TEMP,
]
print("Running single test:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, check=False, cwd=str(repo_base))
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("Return code:", result.returncode)

# Optional: copy single-run results to a separate file
results_file = repo_base / "results.txt"
if results_file.exists():
    dst = repo_base / "eomt" / "results_single_test.txt"
    shutil.copyfile(results_file, dst)
    print("Saved single-test results to:", dst)
else:
    print("results.txt not found after single run.")


## Run MSP batch
Logs to `batch_output_msp.txt`, results to `eomt/results_msp.txt`.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")

CKPT = str(repo_base / "checkpoints" / "eomt_cityscapes.bin")
BASE = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset")
METHOD = "msp"
LOG = f"batch_output_{METHOD}.txt"

print("Running MSP batch evaluation...")
print("Checkpoint:", CKPT)
print("Base dir:", BASE)
print("\n" + "="*60)

# Stream output in real-time while saving to file
with open(LOG, "w", encoding="utf-8") as logf:
    process = subprocess.Popen([
        "python3", str(repo_base / "eomt" / "run_maxentropy_batch.py"),
        "--ckpt", CKPT,
        "--base-dir", BASE,
        "--method", METHOD,
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(repo_base), bufsize=1)
    
    for line in process.stdout:
        print(line, end='')  # Print to console
        logf.write(line)     # Write to file
    
    process.wait()

print("="*60)
print("Saved log to:", LOG)

# Move results.txt to final destination
results_file = repo_base / "results.txt"
if results_file.exists():
    dst = repo_base / "eomt" / f"results_{METHOD}.txt"
    shutil.copyfile(results_file, dst)
    print("Saved results to:", dst)
    results_file.unlink()  # Remove results.txt for next run
else:
    print("No results.txt found after run.")

In [ ]:
import os

# View MSP log to diagnose errors
log_file = "batch_output_msp.txt"
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        content = f.read()
    print(f"=== {log_file} ===")
    print(content)
else:
    print(f"{log_file} not found")


## Run MaxLogit batch
Logs to `batch_output_maxlogit.txt`, results to `eomt/results_maxlogit.txt`.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")

CKPT = str(repo_base / "checkpoints" / "eomt_cityscapes.bin")
BASE = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset")
METHOD = "maxlogit"
LOG = f"batch_output_{METHOD}.txt"

print("Running MaxLogit batch evaluation...")
print("Checkpoint:", CKPT)
print("Base dir:", BASE)
print("\n" + "="*60)

# Stream output in real-time while saving to file
with open(LOG, "w", encoding="utf-8") as logf:
    process = subprocess.Popen([
        "python3", str(repo_base / "eomt" / "run_maxentropy_batch.py"),
        "--ckpt", CKPT,
        "--base-dir", BASE,
        "--method", METHOD,
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(repo_base), bufsize=1)
    
    for line in process.stdout:
        print(line, end='')  # Print to console
        logf.write(line)     # Write to file
    
    process.wait()

print("="*60)
print("Saved log to:", LOG)

# Move results.txt to final destination
results_file = repo_base / "results.txt"
if results_file.exists():
    dst = repo_base / "eomt" / f"results_{METHOD}.txt"
    shutil.copyfile(results_file, dst)
    print("Saved results to:", dst)
    results_file.unlink()  # Remove results.txt for next run
else:
    print("No results.txt found after run.")

## Run MaxEntropy batch
Logs to `batch_output_maxentropy.txt`, results to `eomt/results_maxentropy.txt`.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")

CKPT = str(repo_base / "checkpoints" / "eomt_cityscapes.bin")
BASE = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset")
METHOD = "maxentropy"
LOG = f"batch_output_{METHOD}.txt"

print("Running MaxEntropy batch evaluation...")
print("Checkpoint:", CKPT)
print("Base dir:", BASE)
print("\n" + "="*60)

# Stream output in real-time while saving to file
with open(LOG, "w", encoding="utf-8") as logf:
    process = subprocess.Popen([
        "python3", str(repo_base / "eomt" / "run_maxentropy_batch.py"),
        "--ckpt", CKPT,
        "--base-dir", BASE,
        "--method", METHOD,
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(repo_base), bufsize=1)
    
    for line in process.stdout:
        print(line, end='')  # Print to console
        logf.write(line)     # Write to file
    
    process.wait()

print("="*60)
print("Saved log to:", LOG)

# Move results.txt to final destination
results_file = repo_base / "results.txt"
if results_file.exists():
    dst = repo_base / "eomt" / f"results_{METHOD}.txt"
    shutil.copyfile(results_file, dst)
    print("Saved results to:", dst)
    results_file.unlink()  # Remove results.txt for next run
else:
    print("No results.txt found after run.")

## Run RBA batch
Logs to `batch_output_rba.txt`, results to `eomt/results_rba.txt`.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")

CKPT = str(repo_base / "checkpoints" / "eomt_cityscapes.bin")
BASE = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset")
METHOD = "rba"
LOG = f"batch_output_{METHOD}.txt"

print("Running RBA batch evaluation...")
print("Checkpoint:", CKPT)
print("Base dir:", BASE)
print("\n" + "="*60)

# Stream output in real-time while saving to file
with open(LOG, "w", encoding="utf-8") as logf:
    process = subprocess.Popen([
        "python3", str(repo_base / "eomt" / "run_maxentropy_batch.py"),
        "--ckpt", CKPT,
        "--base-dir", BASE,
        "--method", METHOD,
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(repo_base), bufsize=1)
    
    for line in process.stdout:
        print(line, end='')  # Print to console
        logf.write(line)     # Write to file
    
    process.wait()

print("="*60)
print("Saved log to:", LOG)

# Move results.txt to final destination
results_file = repo_base / "results.txt"
if results_file.exists():
    dst = repo_base / "eomt" / f"results_{METHOD}.txt"
    shutil.copyfile(results_file, dst)
    print("Saved results to:", dst)
    results_file.unlink()  # Remove results.txt for next run
else:
    print("No results.txt found after run.")

# temperature scaling

In [ ]:
# --- BATCH EVALUATION ---
import os
import subprocess
import datetime
import shutil
from pathlib import Path

# ================= CONFIGURATION =================
# Base path of the repository
repo_base = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject")

# Checkpoint to evaluate (Change this path if the epoch changes)
CKPT = str(repo_base / "outputs" / "checkpoints" / "epoch3.ckpt")

# Lists of parameters to iterate over
DATASETS = [
    "RoadAnomaly",
    "RoadAnomaly21",
    "RoadObsticle21",
    "fs_static",
    "FS_LostFound_full"
]

TEMPS = [0.5, 0.75, 1, 1.1, 1.2, 2]
METHODS = ["msp", "maxlogit", "maxentropy", "rba"] 

# Enable LoRA if the checkpoint requires it
USE_LORA = True 
# ==================================================

# 1. Prepare output folder
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = repo_base / "eomt" / f"batch_results_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f" STARTING BATCH EVALUATION")
print(f" Results will be saved to: {output_dir}")
print(f" Checkpoint: {CKPT}")
print("="*60)

# 2. Main loop
for method in METHODS:
    for dataset in DATASETS:
        # Build the pattern to find images for the current dataset
        input_pattern = str(repo_base / "dataset" / "anomaly" / "Validation_Dataset" / dataset / "images" / "*.*")
        
        for temp in TEMPS:
            # Unique identifier for this run
            run_id = f"{method}_{dataset}_T{temp}"
            print(f"\n▶ Running: {run_id} ...", end=" ")

            # A. PREVENTIVE CLEANUP
            # Remove any leftover results.txt in the root to avoid false positives
            default_result_file = repo_base / "results.txt"
            if default_result_file.exists():
                default_result_file.unlink()

            # B. BUILD COMMAND
            cmd = [
                "python3", str(repo_base / "eomt" / "evalAnomaly.py"),
                "--ckpt", CKPT,
                "--input", input_pattern,
                "--method", method,
                "--temp", str(temp)
            ]
            if USE_LORA:
                cmd.append("--lora")

            # C. EXECUTION
            # Capture stdout (prints) and stderr (errors)
            result = subprocess.run(
                cmd, 
                capture_output=True, 
                text=True, 
                cwd=str(repo_base) # Run from project root
            )

            # D. SAVE SYSTEM LOG (What the console printed)
            log_file = output_dir / f"LOG_{run_id}.txt"
            with open(log_file, "w", encoding="utf-8") as f:
                f.write(f"COMMAND: {' '.join(cmd)}\n\n")
                f.write("=== STDOUT ===\n")
                f.write(result.stdout)
                f.write("\n\n=== STDERR ===\n")
                f.write(result.stderr)

            # E. HANDLE RESULTS FILE
            # If the called script succeeded, it should have created results.txt in the root
            if default_result_file.exists():
                # Rename and move into the organized folder
                final_result_file = output_dir / f"RESULT_{run_id}.txt"
                shutil.move(str(default_result_file), str(final_result_file))
                print(" DONE")
            else:
                # If the file is missing, something went wrong in the called script
                print(" ERROR (No results.txt generated)")
                print(f"   See the log for details: {log_file.name}")

print("\n" + "="*60)
print(f" ALL DONE.")
print(f" You can find all files here: {output_dir}")